<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.2/blob/main/Integrated_N%2B_Caption_Generator_from_M_I_Features_and_ILT_Dynamics_2605.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Google Colab COMPLETE CODE Part 1/2
# Correct N+ caption generator using Copolymer_MI_caption_features.csv
#
# Required:
#   1) Copolymer_MI_caption_features.csv
#   2) ilt_summary_ridge_improved.csv
#
# Optional:
#   3) Copolymer_C_numeric_features.csv
#   4) Copolymer_Captions_T2smallmiddlestrong.csv
# ============================================================

!pip -q install pandas numpy openpyxl xlsxwriter

import os
import re
import glob
import numpy as np
import pandas as pd

# ============================================================
# 0. Settings
# ============================================================

PROCESS_SOLVENT = "DMSO"
PROCESS_TEMP_C = 56
PROCESS_TIME_H = 16
PROCESS_INITIATOR = "AIBN"
PROCESS_INITIATOR_FRACTION = 0.03

OUT_CSV  = "/content/Copolymer_Captions_optimized_Nplus_labels_MI_features_fixed.csv"
OUT_XLSX = "/content/Copolymer_Captions_optimized_Nplus_labels_MI_features_fixed.xlsx"

# ============================================================
# 1. File utilities
# ============================================================

def read_csv_flexible(path):
    last_err = None
    for enc in ["utf-8-sig", "utf-8", "cp932", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"Loaded: {os.path.basename(path)} | encoding={enc} | shape={df.shape}")
            return df
        except Exception as e:
            last_err = e
    raise last_err

def find_one(patterns, exclude_keywords=None):
    exclude_keywords = exclude_keywords or []
    hits = []
    for p in patterns:
        hits.extend(glob.glob(p))
    hits = sorted(set(hits))

    clean_hits = []
    for h in hits:
        low = os.path.basename(h).lower()
        if any(k.lower() in low for k in exclude_keywords):
            continue
        clean_hits.append(h)

    return clean_hits[0] if clean_hits else None

def upload_if_needed():
    csvs = glob.glob("/content/*.csv")
    if len(csvs) == 0:
        from google.colab import files
        files.upload()

upload_if_needed()

# ============================================================
# 2. Correct file detection
# ============================================================

MI_FEATURE_PATH = find_one(
    [
        "/content/Copolymer_MI_caption_features.csv",
        "/content/*Copolymer_MI_caption_features*.csv",
        "/content/*mi_caption_features*.csv",
    ],
    exclude_keywords=["summary"]
)

ILT_PATH = find_one([
    "/content/ilt_summary_ridge_improved.csv",
    "/content/*ilt_summary_ridge*.csv",
])

CNUM_PATH = find_one([
    "/content/Copolymer_C_numeric_features.csv",
    "/content/*C_numeric_features*.csv",
])

CAPTION_PATH = find_one(
    [
        "/content/Copolymer_Captions_T2smallmiddlestrong.csv",
        "/content/*Copolymer_Captions_T2*.csv",
        "/content/*captions*.csv",
    ],
    exclude_keywords=[
        "mi_caption_summary",
        "mi_caption_features",
        "optimized_nplus",
    ]
)

print("\nDetected files")
print("MI_FEATURE_PATH:", MI_FEATURE_PATH)
print("ILT_PATH       :", ILT_PATH)
print("CNUM_PATH      :", CNUM_PATH)
print("CAPTION_PATH   :", CAPTION_PATH)

if MI_FEATURE_PATH is None:
    raise FileNotFoundError(
        "Copolymer_MI_caption_features.csv was not found. "
        "Run code3 first to generate this file."
    )

if "summary" in os.path.basename(MI_FEATURE_PATH).lower():
    raise ValueError(
        "Copolymer_MI_caption_summary.csv was detected, but this is a text summary file. "
        "Use Copolymer_MI_caption_features.csv instead."
    )

if ILT_PATH is None:
    raise FileNotFoundError(
        "ilt_summary_ridge_improved.csv was not found."
    )

# ============================================================
# 3. Name utilities
# ============================================================

def normalize_name_keep_suffix(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = s.replace("4VBA", "VBA")
    s = s.replace("MEDSH", "MEDSAH")
    s = s.replace("TECL2", "TECL")
    s = s.replace("-", "_")
    s = re.sub(r"\s+", "", s)
    return s

def core_polymer_name(x):
    s = normalize_name_keep_suffix(x)
    parts = s.split("_")
    if len(parts) >= 3:
        return "_".join(parts[:3])
    return s

def parse_copolymer_name(x):
    core = core_polymer_name(x)
    parts = core.split("_")
    if len(parts) < 3:
        return np.nan, np.nan, np.nan
    return parts[0], parts[1], parts[2]

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def safe_float(x):
    try:
        if pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def fmt_num(x, digits=3):
    x = safe_float(x)
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}g}"

def fmt_fixed(x, digits=3):
    x = safe_float(x)
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}".rstrip("0").rstrip(".")

def extract_ratio_from_text(text):
    if pd.isna(text):
        return None

    txt = str(text)
    m = re.search(r"\(([\d\.]+)\s*:\s*([\d\.]+)\s*:\s*([\d\.]+)\)", txt)
    if m:
        return float(m.group(1)), float(m.group(2)), float(m.group(3))

    perc = re.findall(r"(\d+(?:\.\d+)?)\s*%", txt)
    if len(perc) >= 3:
        return float(perc[0]) / 100, float(perc[1]) / 100, float(perc[2]) / 100

    return None

def join_parts(parts):
    out = []
    for p in parts:
        if p is None:
            continue
        s = str(p).strip()
        if s == "" or s.lower() == "nan":
            continue
        out.append(s)
    return " ".join(out).replace("  ", " ").strip()

# ============================================================
# 4. Chemistry dictionaries
# ============================================================

HYDROPHILIC_INFO = {
    "AMPS":   {"tag": "anionic-sulfonate",     "short": "anionic sulfonate hydrophilic units"},
    "pSSA":   {"tag": "aromatic-sulfonate",    "short": "aromatic sulfonate hydrophilic units"},
    "MEDSAH": {"tag": "zwitterionic",         "short": "zwitterionic hydrophilic units"},
    "VBA":    {"tag": "aromatic-carboxylate", "short": "aromatic carboxylate hydrophilic units"},
    "NIPAM":  {"tag": "amide-hydrophilic",    "short": "amide-bearing hydrophilic units"},
    "HEA":    {"tag": "hydroxyl-hydrophilic", "short": "hydroxyl-bearing hydrophilic units"},
}

HYDROPHOBIC_INFO = {
    "TFEMA": {"tag": "fluorinated-hydrophobic", "short": "fluorinated hydrophobic units"},
    "HMA":   {"tag": "long-alkyl-hydrophobic",  "short": "long-alkyl hydrophobic units"},
    "HA":    {"tag": "alkyl-hydrophobic",       "short": "alkyl hydrophobic units"},
}

CROSSLINKER_INFO = {
    "DICL": {"tag": "difunctional-crosslinker",    "topology": "more weakly crosslinked topology",     "functionality": 2},
    "TRCL": {"tag": "trifunctional-crosslinker",   "topology": "intermediately crosslinked topology", "functionality": 3},
    "TECL": {"tag": "tetrafunctional-crosslinker", "topology": "highly crosslinked topology",        "functionality": 4},
}

HYDROPHILIC_FEATURES = {
    "AMPS":   dict(has_sulfonate=1, has_carboxylate=0, has_amide=0, has_hydroxyl=0, has_aromatic_hydrophilic=0, has_ionic_group=1, hydrophilic_polarity_score=3),
    "pSSA":   dict(has_sulfonate=1, has_carboxylate=0, has_amide=0, has_hydroxyl=0, has_aromatic_hydrophilic=1, has_ionic_group=1, hydrophilic_polarity_score=3),
    "MEDSAH": dict(has_sulfonate=1, has_carboxylate=0, has_amide=0, has_hydroxyl=0, has_aromatic_hydrophilic=0, has_ionic_group=1, hydrophilic_polarity_score=3),
    "VBA":    dict(has_sulfonate=0, has_carboxylate=1, has_amide=0, has_hydroxyl=0, has_aromatic_hydrophilic=1, has_ionic_group=1, hydrophilic_polarity_score=2),
    "NIPAM":  dict(has_sulfonate=0, has_carboxylate=0, has_amide=1, has_hydroxyl=0, has_aromatic_hydrophilic=0, has_ionic_group=0, hydrophilic_polarity_score=1),
    "HEA":    dict(has_sulfonate=0, has_carboxylate=0, has_amide=0, has_hydroxyl=1, has_aromatic_hydrophilic=0, has_ionic_group=0, hydrophilic_polarity_score=1),
}

HYDROPHOBIC_FEATURES = {
    "TFEMA": dict(has_fluorinated_group=1, has_long_alkyl_group=0, has_alkyl_group=1, hydrophobicity_proxy=3, fluorinated_hydrophobicity_proxy=3, alkyl_hydrophobicity_proxy=1),
    "HMA":   dict(has_fluorinated_group=0, has_long_alkyl_group=1, has_alkyl_group=1, hydrophobicity_proxy=3, fluorinated_hydrophobicity_proxy=0, alkyl_hydrophobicity_proxy=3),
    "HA":    dict(has_fluorinated_group=0, has_long_alkyl_group=0, has_alkyl_group=1, hydrophobicity_proxy=2, fluorinated_hydrophobicity_proxy=0, alkyl_hydrophobicity_proxy=2),
}

# ============================================================
# 5. Load MI feature and ILT files
# ============================================================

mi_feat = read_csv_flexible(MI_FEATURE_PATH)
ilt_df = read_csv_flexible(ILT_PATH)

mi_feat.columns = [str(c).strip() for c in mi_feat.columns]
ilt_df.columns = [str(c).strip() for c in ilt_df.columns]

name_candidates = [
    "Copolymer_Name",
    "Copolymer_Name_exact",
    "Copolymer_Name_core",
    "Sample_Name",
    "Sample",
    "Name",
    "Original_name",
]

mi_name_col = first_existing_col(mi_feat, name_candidates)
if mi_name_col is None:
    raise KeyError(f"No name column found in MI feature file. columns={mi_feat.columns.tolist()}")

# ============================================================
# 6. Base sample table from MI features
# ============================================================

base = pd.DataFrame()
base["Copolymer_Name"] = mi_feat[mi_name_col].astype(str)
base["Copolymer_Name_exact"] = base["Copolymer_Name"].apply(normalize_name_keep_suffix)
base["Copolymer_Name_core"] = base["Copolymer_Name"].apply(core_polymer_name)

parsed = base["Copolymer_Name_core"].apply(parse_copolymer_name)
base["hydrophilic_monomer"] = parsed.apply(lambda x: x[0])
base["hydrophobic_monomer"] = parsed.apply(lambda x: x[1])
base["crosslinker"] = parsed.apply(lambda x: x[2])

base = base.drop_duplicates("Copolymer_Name_exact").reset_index(drop=True)

# ============================================================
# 7. Add composition ratios
# ============================================================

def add_ratios(base):
    out = base.copy()

    out["hydrophilic_fraction"] = np.nan
    out["hydrophobic_fraction"] = np.nan
    out["crosslinker_fraction"] = np.nan

    ratio_cols = [
        "hydrophilic_fraction",
        "hydrophobic_fraction",
        "crosslinker_fraction",
    ]

    # 7-1. Prefer C numeric file
    if CNUM_PATH is not None:
        cnum = read_csv_flexible(CNUM_PATH)
        cnum.columns = [str(c).strip() for c in cnum.columns]
        c_name_col = first_existing_col(cnum, name_candidates)

        if c_name_col is not None and all(c in cnum.columns for c in ratio_cols):
            cnum["Copolymer_Name_exact"] = cnum[c_name_col].apply(normalize_name_keep_suffix)
            cnum["Copolymer_Name_core"] = cnum[c_name_col].apply(core_polymer_name)

            exact = cnum[["Copolymer_Name_exact"] + ratio_cols].drop_duplicates("Copolymer_Name_exact")
            out = out.drop(columns=ratio_cols, errors="ignore").merge(
                exact,
                on="Copolymer_Name_exact",
                how="left"
            )

            missing = out["hydrophilic_fraction"].isna()
            if missing.any():
                core = (
                    cnum[["Copolymer_Name_core"] + ratio_cols]
                    .groupby("Copolymer_Name_core", as_index=False)
                    .mean(numeric_only=True)
                )
                fill = out.loc[missing, ["Copolymer_Name_core"]].merge(
                    core,
                    on="Copolymer_Name_core",
                    how="left"
                )
                for c in ratio_cols:
                    out.loc[missing, c] = fill[c].values

            return out

    # 7-2. Fallback: extract from caption file
    if CAPTION_PATH is not None:
        cap = read_csv_flexible(CAPTION_PATH)
        cap.columns = [str(c).strip() for c in cap.columns]
        cap_name_col = first_existing_col(cap, name_candidates)

        if cap_name_col is not None:
            cap["Copolymer_Name_exact"] = cap[cap_name_col].apply(normalize_name_keep_suffix)
            cap["Copolymer_Name_core"] = cap[cap_name_col].apply(core_polymer_name)

            text_cols = [
                "caption_T2_small",
                "Caption_T2_middle",
                "Caption_T2_strong",
                "caption",
                "Caption",
                "text",
                "Text",
                "N",
                "N+S",
                "N+SCMIDP",
            ]

            ratios = []
            for _, r in cap.iterrows():
                ratio = None
                for c in text_cols:
                    if c in cap.columns:
                        ratio = extract_ratio_from_text(r[c])
                        if ratio is not None:
                            break
                ratios.append(ratio)

            cap["_ratio_tuple"] = ratios
            cap = cap[cap["_ratio_tuple"].notna()].copy()

            if len(cap) > 0:
                cap["hydrophilic_fraction"] = cap["_ratio_tuple"].apply(lambda x: x[0])
                cap["hydrophobic_fraction"] = cap["_ratio_tuple"].apply(lambda x: x[1])
                cap["crosslinker_fraction"] = cap["_ratio_tuple"].apply(lambda x: x[2])

                exact = cap[["Copolymer_Name_exact"] + ratio_cols].drop_duplicates("Copolymer_Name_exact")
                out = out.drop(columns=ratio_cols, errors="ignore").merge(
                    exact,
                    on="Copolymer_Name_exact",
                    how="left"
                )

                missing = out["hydrophilic_fraction"].isna()
                if missing.any():
                    core = (
                        cap[["Copolymer_Name_core"] + ratio_cols]
                        .groupby("Copolymer_Name_core", as_index=False)
                        .mean(numeric_only=True)
                    )
                    fill = out.loc[missing, ["Copolymer_Name_core"]].merge(
                        core,
                        on="Copolymer_Name_core",
                        how="left"
                    )
                    for c in ratio_cols:
                        out.loc[missing, c] = fill[c].values

    return out

base = add_ratios(base)

# ============================================================
# 8. Generate chemistry numeric features
# ============================================================

def get_feat(key, dictionary):
    if key in dictionary:
        return dictionary[key].copy()
    example = next(iter(dictionary.values()))
    return {k: np.nan for k in example.keys()}

rows = []

for _, row in base.iterrows():
    h = row["hydrophilic_monomer"]
    p = row["hydrophobic_monomer"]
    c = row["crosslinker"]

    rec = row.to_dict()
    rec.update(get_feat(h, HYDROPHILIC_FEATURES))
    rec.update(get_feat(p, HYDROPHOBIC_FEATURES))
    rec["crosslinker_functionality"] = CROSSLINKER_INFO.get(c, {"functionality": np.nan})["functionality"]
    rec["crosslink_density_proxy"] = rec["crosslinker_functionality"]

    rows.append(rec)

df = pd.DataFrame(rows)

df["weighted_polarity_score"] = df["hydrophilic_fraction"] * df["hydrophilic_polarity_score"]
df["weighted_hydrophobicity_score"] = df["hydrophobic_fraction"] * df["hydrophobicity_proxy"]
df["weighted_fluorinated_score"] = df["hydrophobic_fraction"] * df["fluorinated_hydrophobicity_proxy"]
df["weighted_alkyl_score"] = df["hydrophobic_fraction"] * df["alkyl_hydrophobicity_proxy"]
df["weighted_crosslink_density"] = df["crosslinker_fraction"] * df["crosslink_density_proxy"]

df["sulfonate_fraction_proxy"] = df["hydrophilic_fraction"] * df["has_sulfonate"]
df["carboxylate_fraction_proxy"] = df["hydrophilic_fraction"] * df["has_carboxylate"]
df["amide_fraction_proxy"] = df["hydrophilic_fraction"] * df["has_amide"]
df["hydroxyl_fraction_proxy"] = df["hydrophilic_fraction"] * df["has_hydroxyl"]
df["ionic_fraction_proxy"] = df["hydrophilic_fraction"] * df["has_ionic_group"]
df["aromatic_fraction_proxy"] = df["hydrophilic_fraction"] * df["has_aromatic_hydrophilic"]
df["fluorinated_fraction_proxy"] = df["hydrophobic_fraction"] * df["has_fluorinated_group"]
df["alkyl_fraction_proxy"] = df["hydrophobic_fraction"] * df["has_alkyl_group"]
df["long_alkyl_fraction_proxy"] = df["hydrophobic_fraction"] * df["has_long_alkyl_group"]

df["hydrophilic_to_hydrophobic_ratio"] = (
    df["hydrophilic_fraction"] / df["hydrophobic_fraction"].replace(0, np.nan)
)
df["crosslink_to_total_monomer_ratio"] = (
    df["crosslinker_fraction"]
    / (df["hydrophilic_fraction"] + df["hydrophobic_fraction"]).replace(0, np.nan)
)
df["amphiphilicity_balance"] = (
    1.0 - np.abs(df["hydrophilic_fraction"] - df["hydrophobic_fraction"])
)
df["network_restriction_proxy"] = (
    df["weighted_crosslink_density"] * df["weighted_polarity_score"].fillna(0)
)
df["hydrophobic_interaction_proxy"] = (
    df["weighted_hydrophobicity_score"]
    + df["weighted_fluorinated_score"]
    + df["weighted_alkyl_score"]
)

# ============================================================
# 9. Merge correct M/I features
# ============================================================

mi = mi_feat.copy()
mi["Copolymer_Name_exact"] = mi[mi_name_col].apply(normalize_name_keep_suffix)
mi["Copolymer_Name_core"] = mi[mi_name_col].apply(core_polymer_name)

required_mi_source_cols = [
    "I_soft_headgroup",
    "I_soft_glycerol",
    "I_soft_alkenyl",
    "I_soft_alkyl_chain",
    "I_dominant_motif",
]

missing_source = [c for c in required_mi_source_cols if c not in mi.columns]
if missing_source:
    raise ValueError(
        "Copolymer_MI_caption_features.csv is missing required columns: "
        + str(missing_source)
        + "\nRun code3 again and use Copolymer_MI_caption_features.csv, not summary."
    )

mi_map = pd.DataFrame()
mi_map["Copolymer_Name_exact"] = mi["Copolymer_Name_exact"]
mi_map["Copolymer_Name_core"] = mi["Copolymer_Name_core"]

# Convert code3 columns to the names expected by caption generator
mi_map["dominant_motif"] = mi["I_dominant_motif"]
mi_map["headgroup_score_norm"] = pd.to_numeric(mi["I_soft_headgroup"], errors="coerce")
mi_map["glycerol_score_norm"] = pd.to_numeric(mi["I_soft_glycerol"], errors="coerce")
mi_map["alkenyl_score_norm"] = pd.to_numeric(mi["I_soft_alkenyl"], errors="coerce")
mi_map["alkyl_chain_score_norm"] = pd.to_numeric(mi["I_soft_alkyl_chain"], errors="coerce")

# Optional text-rich columns
for src, dst in {
    "M_motif_sentence": "M_motif_sentence",
    "I_interaction_sentence": "I_interaction_sentence",
    "caption_fragment_MI": "caption_fragment_MI",
    "caption_short_MI": "caption_short_MI",
    "I_scope": "I_scope",
    "I_dominance_strength": "I_dominance_strength",
    "I_entropy": "I_entropy",
    "I_polar_chain_class": "I_polar_chain_class",
}.items():
    mi_map[dst] = mi[src] if src in mi.columns else np.nan

mi_numeric_cols = [
    "headgroup_score_norm",
    "glycerol_score_norm",
    "alkenyl_score_norm",
    "alkyl_chain_score_norm",
]

mi_text_cols = [
    "dominant_motif",
    "M_motif_sentence",
    "I_interaction_sentence",
    "caption_fragment_MI",
    "caption_short_MI",
    "I_scope",
    "I_dominance_strength",
    "I_entropy",
    "I_polar_chain_class",
]

# exact merge
mi_exact_num = (
    mi_map[["Copolymer_Name_exact"] + mi_numeric_cols]
    .groupby("Copolymer_Name_exact", as_index=False)
    .mean(numeric_only=True)
)

mi_exact_text = (
    mi_map[["Copolymer_Name_exact"] + mi_text_cols]
    .drop_duplicates("Copolymer_Name_exact")
)

mi_exact = mi_exact_text.merge(mi_exact_num, on="Copolymer_Name_exact", how="outer")

df = df.merge(mi_exact, on="Copolymer_Name_exact", how="left")

# core fallback
missing_mi = df[mi_numeric_cols].isna().all(axis=1)

if missing_mi.any():
    mi_core_num = (
        mi_map[["Copolymer_Name_core"] + mi_numeric_cols]
        .groupby("Copolymer_Name_core", as_index=False)
        .mean(numeric_only=True)
    )

    mi_core_text = (
        mi_map[["Copolymer_Name_core"] + mi_text_cols]
        .drop_duplicates("Copolymer_Name_core")
    )

    mi_core = mi_core_text.merge(mi_core_num, on="Copolymer_Name_core", how="outer")

    fill = df.loc[missing_mi, ["Copolymer_Name_core"]].merge(
        mi_core,
        on="Copolymer_Name_core",
        how="left",
        suffixes=("", "_core")
    )

    for c in mi_numeric_cols + mi_text_cols:
        if c in fill.columns:
            df.loc[missing_mi, c] = fill[c].values

motif_score_cols = [
    "headgroup_score_norm",
    "glycerol_score_norm",
    "alkenyl_score_norm",
    "alkyl_chain_score_norm",
]

df["motif_score_sum"] = df[motif_score_cols].sum(axis=1, skipna=True)
df["motif_score_max"] = df[motif_score_cols].max(axis=1, skipna=True)

df["motif_score_second"] = df[motif_score_cols].apply(
    lambda r: sorted(
        [x for x in pd.to_numeric(r, errors="coerce") if pd.notna(x)],
        reverse=True
    )[1]
    if len([x for x in pd.to_numeric(r, errors="coerce") if pd.notna(x)]) >= 2
    else np.nan,
    axis=1,
)

df["motif_breadth_count_0p5"] = df[motif_score_cols].apply(
    lambda r: int(np.sum(pd.to_numeric(r, errors="coerce") >= 0.5))
    if pd.to_numeric(r, errors="coerce").notna().any()
    else np.nan,
    axis=1,
)

df["motif_selectivity_index"] = df["motif_score_max"] - df["motif_score_second"]

print("\nMI merge check:")
print(df[motif_score_cols + ["dominant_motif"]].head())
print(df[motif_score_cols].isna().sum())

# ============================================================
# Google Colab COMPLETE CODE Part 2/2
# Continue after Part 1/2
# ============================================================

# ============================================================
# 10. Merge ILT / D numeric information
# ============================================================

ilt_name_col = first_existing_col(ilt_df, name_candidates)

if ilt_name_col is None:
    raise KeyError(
        f"No name column found in ILT file. columns={ilt_df.columns.tolist()}"
    )

ilt = ilt_df.copy()
ilt["Copolymer_Name_exact"] = ilt[ilt_name_col].apply(normalize_name_keep_suffix)
ilt["Copolymer_Name_core"] = ilt[ilt_name_col].apply(core_polymer_name)

def pick_ilt_col(candidates):
    return first_existing_col(ilt, candidates)

col_logmean = pick_ilt_col(["Weighted_logmean_T2_ms"])
col_width   = pick_ilt_col(["Width_log10T2"])
col_fitr2   = pick_ilt_col(["Fit_R2"])
col_alpha   = pick_ilt_col(["Ridge_alpha"])
col_npeak   = pick_ilt_col(["N_detected_peaks", "N_selected_components"])
col_rawmax  = pick_ilt_col(["Peak_T2_ms_raw_max", "Peak_T2_ms"])

ilt_std = pd.DataFrame()
ilt_std["Copolymer_Name_exact"] = ilt["Copolymer_Name_exact"]
ilt_std["Copolymer_Name_core"] = ilt["Copolymer_Name_core"]

ilt_std["Weighted_logmean_T2_ms"] = ilt[col_logmean] if col_logmean else np.nan
ilt_std["Width_log10T2"] = ilt[col_width] if col_width else np.nan
ilt_std["Fit_R2"] = ilt[col_fitr2] if col_fitr2 else np.nan
ilt_std["Ridge_alpha"] = ilt[col_alpha] if col_alpha else np.nan
ilt_std["N_detected_peaks"] = ilt[col_npeak] if col_npeak else np.nan
ilt_std["Peak_T2_ms_raw_max"] = ilt[col_rawmax] if col_rawmax else np.nan

for i in range(1, 5):
    src = pick_ilt_col([
        f"Peak{i}_T2_ms",
        f"Component{i}_peakT2_ms",
        f"Component{i}_T2_ms",
    ])
    ilt_std[f"Peak{i}_T2_ms"] = ilt[src] if src else np.nan

for std, candidates in {
    "ShortFraction": ["ShortFraction", "short_fraction", "Fraction_short"],
    "MidFraction": ["MidFraction", "middle_fraction", "mid_fraction", "Fraction_mid"],
    "LongFraction": ["LongFraction", "long_fraction", "Fraction_long"],
}.items():
    src = pick_ilt_col(candidates)
    ilt_std[std] = ilt[src] if src else np.nan

for c in ilt_std.columns:
    if c not in ["Copolymer_Name_exact", "Copolymer_Name_core"]:
        ilt_std[c] = pd.to_numeric(ilt_std[c], errors="coerce")

no_peak = ilt_std[
    ["Peak1_T2_ms", "Peak2_T2_ms", "Peak3_T2_ms", "Peak4_T2_ms"]
].isna().all(axis=1)

ilt_std.loc[no_peak, "Peak1_T2_ms"] = ilt_std.loc[
    no_peak, "Peak_T2_ms_raw_max"
]

infer_n = ilt_std[
    ["Peak1_T2_ms", "Peak2_T2_ms", "Peak3_T2_ms", "Peak4_T2_ms"]
].notna().sum(axis=1)

ilt_std["N_detected_peaks"] = ilt_std["N_detected_peaks"].fillna(infer_n)

ilt_numeric_cols = [
    "Weighted_logmean_T2_ms",
    "Width_log10T2",
    "Fit_R2",
    "Ridge_alpha",
    "N_detected_peaks",
    "Peak_T2_ms_raw_max",
    "Peak1_T2_ms",
    "Peak2_T2_ms",
    "Peak3_T2_ms",
    "Peak4_T2_ms",
    "ShortFraction",
    "MidFraction",
    "LongFraction",
]

ilt_exact = (
    ilt_std[["Copolymer_Name_exact"] + ilt_numeric_cols]
    .groupby("Copolymer_Name_exact", as_index=False)
    .mean(numeric_only=True)
)

df = df.merge(ilt_exact, on="Copolymer_Name_exact", how="left")

missing_d = df[ilt_numeric_cols].isna().all(axis=1)

if missing_d.any():
    ilt_core = (
        ilt_std[["Copolymer_Name_core"] + ilt_numeric_cols]
        .groupby("Copolymer_Name_core", as_index=False)
        .mean(numeric_only=True)
    )

    fill = df.loc[missing_d, ["Copolymer_Name_core"]].merge(
        ilt_core,
        on="Copolymer_Name_core",
        how="left"
    )

    for c in ilt_numeric_cols:
        if c in fill.columns:
            df.loc[missing_d, c] = fill[c].values

# ============================================================
# 11. Process numeric information
# ============================================================

df["process_temperature_C"] = PROCESS_TEMP_C
df["process_time_h"] = PROCESS_TIME_H
df["initiator_fraction"] = PROCESS_INITIATOR_FRACTION
df["process_solvent_DMSO"] = 1

# ============================================================
# 12. Text helper functions
# ============================================================

def ratio_text(row):
    a = row.get("hydrophilic_fraction", np.nan)
    b = row.get("hydrophobic_fraction", np.nan)
    c = row.get("crosslinker_fraction", np.nan)

    if pd.isna(a) or pd.isna(b) or pd.isna(c):
        return "NA"

    return f"{fmt_fixed(a)}:{fmt_fixed(b)}:{fmt_fixed(c)}"

def structure_phrase(row):
    return f"{row['Copolymer_Name_exact']} ({ratio_text(row)})"

def chemistry_text(row):
    h = row.get("hydrophilic_monomer", "")
    p = row.get("hydrophobic_monomer", "")
    c = row.get("crosslinker", "")

    hi = HYDROPHILIC_INFO.get(
        h,
        {"tag": "hydrophilic", "short": "hydrophilic units"},
    )
    pi = HYDROPHOBIC_INFO.get(
        p,
        {"tag": "hydrophobic", "short": "hydrophobic units"},
    )
    ci = CROSSLINKER_INFO.get(
        c,
        {"tag": "crosslinker", "topology": "crosslinked topology"},
    )

    return (
        f"Chemistry: {hi['tag']}, {pi['tag']}, and {ci['tag']}. "
        f"The copolymer combines {hi['short']} with {pi['short']} "
        f"in a {ci['topology']}."
    )

def normalize_motif_label(x):
    s = str(x).strip().lower() if pd.notna(x) else "unknown"

    mapping = {
        "head": "headgroup",
        "headgroup": "headgroup",
        "polar_headgroup": "headgroup",
        "head-group": "headgroup",
        "glycerol": "glycerol",
        "glycerol_backbone": "glycerol",
        "glycerol-backbone": "glycerol",
        "alkenyl": "alkenyl",
        "olefinic": "alkenyl",
        "vinyl": "alkenyl",
        "alkyl": "alkyl_chain",
        "alkyl_chain": "alkyl_chain",
        "alkyl-chain": "alkyl_chain",
        "chain": "alkyl_chain",
    }

    return mapping.get(s, s if s else "unknown")

def motif_text(row):
    # Prefer text-rich M sentence from code3.
    m_sentence = row.get("M_motif_sentence", np.nan)

    if pd.notna(m_sentence) and str(m_sentence).strip() != "":
        return f"Motif: {str(m_sentence).strip()}"

    m = normalize_motif_label(row.get("dominant_motif", np.nan))

    return (
        f"Motif: the dominant lipid-associated motif is {m}. "
        f"Motif scores are headgroup={fmt_num(row.get('headgroup_score_norm'))}, "
        f"glycerol={fmt_num(row.get('glycerol_score_norm'))}, "
        f"alkenyl={fmt_num(row.get('alkenyl_score_norm'))}, "
        f"and alkyl-chain={fmt_num(row.get('alkyl_chain_score_norm'))}."
    )

def interaction_text(row):
    # Prefer text-rich I sentence from code3.
    i_sentence = row.get("I_interaction_sentence", np.nan)

    if pd.notna(i_sentence) and str(i_sentence).strip() != "":
        return f"Interaction: {str(i_sentence).strip()}"

    breadth = safe_float(row.get("motif_breadth_count_0p5"))
    selectivity = safe_float(row.get("motif_selectivity_index"))
    max_score = safe_float(row.get("motif_score_max"))

    if pd.isna(breadth):
        scope = "unknown motif breadth"
    elif breadth >= 3:
        scope = "broad motif perturbation"
    elif breadth == 2:
        scope = "dual-motif perturbation"
    else:
        scope = "localized motif perturbation"

    return (
        f"Interaction: the lipid-polymer interaction shows {scope}, "
        f"with maximum motif score={fmt_num(max_score)} "
        f"and selectivity index={fmt_num(selectivity)}."
    )

def peak_list(row):
    peaks = []
    for i in range(1, 5):
        v = safe_float(row.get(f"Peak{i}_T2_ms", np.nan))
        if not pd.isna(v):
            peaks.append(v)
    return peaks

def peak_regime(peaks):
    if len(peaks) == 0:
        return "unknown-T2-regime"
    if max(peaks) < 10:
        return "short-T2-dominant"
    if max(peaks) < 100:
        return "intermediate-T2-extended"
    return "long-T2-containing"

def width_label(x):
    x = safe_float(x)

    if pd.isna(x):
        return "unknown-width"
    if x < 0.35:
        return "narrow-distribution"
    if x < 0.70:
        return "moderate-distribution"
    return "broad-distribution"

def peak_count_label(x):
    x = safe_float(x)

    if pd.isna(x):
        return "unknown-component"

    n = int(round(x))

    if n <= 1:
        return "single-component"
    if n == 2:
        return "two-component"
    if n == 3:
        return "three-component"

    return "multi-component"

def dynamics_text(row):
    peaks = peak_list(row)
    peak_txt = "NA" if len(peaks) == 0 else "/".join(fmt_num(p) for p in peaks)

    return (
        f"Dynamics: ILT-derived relaxation is {peak_regime(peaks)}, "
        f"{width_label(row.get('Width_log10T2'))}, "
        f"and {peak_count_label(row.get('N_detected_peaks'))}. "
        f"Major T2 peaks are {peak_txt} ms, "
        f"weighted log-mean T2={fmt_num(row.get('Weighted_logmean_T2_ms'))} ms, "
        f"and width(log10T2)={fmt_num(row.get('Width_log10T2'))}."
    )

def process_text(row):
    return (
        f"Process: polymerization was performed in {PROCESS_SOLVENT} "
        f"at {PROCESS_TEMP_C} °C for {PROCESS_TIME_H} h "
        f"using {PROCESS_INITIATOR}."
    )

def number_sentence(row):
    peaks = peak_list(row)
    peak_txt = "NA" if len(peaks) == 0 else "/".join(fmt_num(p) for p in peaks)

    return (
        "Numeric features: "
        f"composition hydrophilic={fmt_num(row.get('hydrophilic_fraction'))}, "
        f"hydrophobic={fmt_num(row.get('hydrophobic_fraction'))}, "
        f"crosslinker={fmt_num(row.get('crosslinker_fraction'))}; "
        f"chemistry polarity={fmt_num(row.get('weighted_polarity_score'))}, "
        f"hydrophobicity={fmt_num(row.get('weighted_hydrophobicity_score'))}, "
        f"fluorinated={fmt_num(row.get('fluorinated_fraction_proxy'))}, "
        f"aromatic={fmt_num(row.get('aromatic_fraction_proxy'))}, "
        f"ionic={fmt_num(row.get('ionic_fraction_proxy'))}, "
        f"crosslink_density_proxy={fmt_num(row.get('weighted_crosslink_density'))}; "
        f"motif scores headgroup={fmt_num(row.get('headgroup_score_norm'))}, "
        f"glycerol={fmt_num(row.get('glycerol_score_norm'))}, "
        f"alkenyl={fmt_num(row.get('alkenyl_score_norm'))}, "
        f"alkyl_chain={fmt_num(row.get('alkyl_chain_score_norm'))}; "
        f"interaction max={fmt_num(row.get('motif_score_max'))}, "
        f"breadth={fmt_num(row.get('motif_breadth_count_0p5'))}, "
        f"selectivity={fmt_num(row.get('motif_selectivity_index'))}; "
        f"dynamics peaks={peak_txt} ms, "
        f"weighted_logmean_T2={fmt_num(row.get('Weighted_logmean_T2_ms'))} ms, "
        f"width_log10T2={fmt_num(row.get('Width_log10T2'))}, "
        f"N_peaks={fmt_num(row.get('N_detected_peaks'))}; "
        f"process temperature={fmt_num(row.get('process_temperature_C'))} C, "
        f"time={fmt_num(row.get('process_time_h'))} h, "
        f"initiator_fraction={fmt_num(row.get('initiator_fraction'))}."
    )

# ============================================================
# 13. Build condition captions
# ============================================================

def cond_N(row):
    return number_sentence(row)

def cond_NS(row):
    return join_parts([
        f"{structure_phrase(row)}.",
        number_sentence(row),
    ])

def cond_NSC(row):
    return join_parts([
        f"{structure_phrase(row)}.",
        chemistry_text(row),
        number_sentence(row),
    ])

def cond_NSM(row):
    return join_parts([
        f"{structure_phrase(row)}.",
        motif_text(row),
        number_sentence(row),
    ])

def cond_NSI(row):
    return join_parts([
        f"{structure_phrase(row)}.",
        interaction_text(row),
        number_sentence(row),
    ])

def cond_NSD(row):
    return join_parts([
        f"{structure_phrase(row)}.",
        dynamics_text(row),
        number_sentence(row),
    ])

def cond_NSP(row):
    return join_parts([
        f"{structure_phrase(row)}.",
        process_text(row),
        number_sentence(row),
    ])

def cond_NSCMIDP(row):
    return join_parts([
        f"{structure_phrase(row)}.",
        chemistry_text(row),
        motif_text(row),
        interaction_text(row),
        dynamics_text(row),
        process_text(row),
        number_sentence(row),
    ])

df["N"] = df.apply(cond_N, axis=1)
df["N+S"] = df.apply(cond_NS, axis=1)
df["N+SC"] = df.apply(cond_NSC, axis=1)
df["N+SM"] = df.apply(cond_NSM, axis=1)
df["N+SI"] = df.apply(cond_NSI, axis=1)
df["N+SD"] = df.apply(cond_NSD, axis=1)
df["N+SP"] = df.apply(cond_NSP, axis=1)
df["N+SCMIDP"] = df.apply(cond_NSCMIDP, axis=1)

# ============================================================
# 14. QC
# ============================================================

c_numeric_cols = [
    "hydrophilic_fraction",
    "hydrophobic_fraction",
    "crosslinker_fraction",
    "weighted_polarity_score",
    "weighted_hydrophobicity_score",
    "weighted_crosslink_density",
]

mi_numeric_cols_qc = [
    "headgroup_score_norm",
    "glycerol_score_norm",
    "alkenyl_score_norm",
    "alkyl_chain_score_norm",
    "motif_score_max",
    "motif_breadth_count_0p5",
]

mi_text_cols_qc = [
    "M_motif_sentence",
    "I_interaction_sentence",
    "caption_fragment_MI",
]

d_numeric_cols = [
    "Peak1_T2_ms",
    "Weighted_logmean_T2_ms",
    "Width_log10T2",
    "N_detected_peaks",
]

p_numeric_cols = [
    "process_temperature_C",
    "process_time_h",
    "initiator_fraction",
]

df["QC_missing_C_numeric"] = df[c_numeric_cols].isna().sum(axis=1)
df["QC_missing_MI_numeric"] = df[mi_numeric_cols_qc].isna().sum(axis=1)
df["QC_missing_MI_text"] = df[mi_text_cols_qc].isna().sum(axis=1)
df["QC_missing_D_numeric"] = df[d_numeric_cols].isna().sum(axis=1)
df["QC_missing_P_numeric"] = df[p_numeric_cols].isna().sum(axis=1)

df["QC_total_missing_core_numeric"] = (
    df["QC_missing_C_numeric"]
    + df["QC_missing_MI_numeric"]
    + df["QC_missing_D_numeric"]
    + df["QC_missing_P_numeric"]
)

print("\nQC summary:")
print(df[[
    "QC_missing_C_numeric",
    "QC_missing_MI_numeric",
    "QC_missing_MI_text",
    "QC_missing_D_numeric",
    "QC_missing_P_numeric",
    "QC_total_missing_core_numeric",
]].sum())

print("\nMI numeric QC distribution:")
print(df["QC_missing_MI_numeric"].value_counts(dropna=False).sort_index())

print("\nMI text QC distribution:")
print(df["QC_missing_MI_text"].value_counts(dropna=False).sort_index())

print("\nRows with missing MI numeric:")
display(
    df.loc[
        df["QC_missing_MI_numeric"] > 0,
        [
            "Copolymer_Name",
            "Copolymer_Name_exact",
            "Copolymer_Name_core",
            "dominant_motif",
            "headgroup_score_norm",
            "glycerol_score_norm",
            "alkenyl_score_norm",
            "alkyl_chain_score_norm",
            "QC_missing_MI_numeric",
        ],
    ]
)

if df["QC_missing_MI_numeric"].sum() > 0:
    print(
        "\nWARNING: Some MI numeric values are still missing. "
        "Check Copolymer_MI_caption_features.csv and name matching."
    )
else:
    print("\nOK: QC_missing_MI_numeric is 0 for all rows.")

# ============================================================
# 15. Save outputs
# ============================================================

caption_cols = [
    "Copolymer_Name",
    "Copolymer_Name_exact",
    "Copolymer_Name_core",
    "hydrophilic_monomer",
    "hydrophobic_monomer",
    "crosslinker",
    "N",
    "N+S",
    "N+SC",
    "N+SM",
    "N+SI",
    "N+SD",
    "N+SP",
    "N+SCMIDP",
]

numeric_cols = [
    # C
    "hydrophilic_fraction",
    "hydrophobic_fraction",
    "crosslinker_fraction",
    "weighted_polarity_score",
    "weighted_hydrophobicity_score",
    "weighted_fluorinated_score",
    "weighted_alkyl_score",
    "weighted_crosslink_density",
    "sulfonate_fraction_proxy",
    "carboxylate_fraction_proxy",
    "amide_fraction_proxy",
    "hydroxyl_fraction_proxy",
    "ionic_fraction_proxy",
    "aromatic_fraction_proxy",
    "fluorinated_fraction_proxy",
    "alkyl_fraction_proxy",
    "long_alkyl_fraction_proxy",
    "hydrophilic_to_hydrophobic_ratio",
    "crosslink_to_total_monomer_ratio",
    "amphiphilicity_balance",
    "network_restriction_proxy",
    "hydrophobic_interaction_proxy",

    # M/I numeric
    "dominant_motif",
    "headgroup_score_norm",
    "glycerol_score_norm",
    "alkenyl_score_norm",
    "alkyl_chain_score_norm",
    "motif_score_sum",
    "motif_score_max",
    "motif_score_second",
    "motif_breadth_count_0p5",
    "motif_selectivity_index",
    "I_scope",
    "I_dominance_strength",
    "I_entropy",
    "I_polar_chain_class",

    # M/I text
    "M_motif_sentence",
    "I_interaction_sentence",
    "caption_short_MI",
    "caption_fragment_MI",

    # D
    "Peak1_T2_ms",
    "Peak2_T2_ms",
    "Peak3_T2_ms",
    "Peak4_T2_ms",
    "Weighted_logmean_T2_ms",
    "Width_log10T2",
    "N_detected_peaks",
    "ShortFraction",
    "MidFraction",
    "LongFraction",
    "Fit_R2",
    "Ridge_alpha",

    # P
    "process_temperature_C",
    "process_time_h",
    "initiator_fraction",
    "process_solvent_DMSO",
]

qc_cols = [
    "QC_missing_C_numeric",
    "QC_missing_MI_numeric",
    "QC_missing_MI_text",
    "QC_missing_D_numeric",
    "QC_missing_P_numeric",
    "QC_total_missing_core_numeric",
]

final_cols = caption_cols + [c for c in numeric_cols if c in df.columns] + qc_cols

out_df = df[final_cols].copy()

out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    out_df.to_excel(writer, index=False, sheet_name="Nplus_captions")

    numeric_only_cols = [c for c in numeric_cols if c in out_df.columns]
    out_df[numeric_only_cols].to_excel(
        writer,
        index=False,
        sheet_name="numeric_features_only"
    )

    qc_detail_cols = [
        "Copolymer_Name",
        "Copolymer_Name_exact",
        "Copolymer_Name_core",
        "dominant_motif",
        "QC_missing_C_numeric",
        "QC_missing_MI_numeric",
        "QC_missing_MI_text",
        "QC_missing_D_numeric",
        "QC_missing_P_numeric",
        "QC_total_missing_core_numeric",
    ]

    out_df[qc_detail_cols].to_excel(
        writer,
        index=False,
        sheet_name="QC_summary"
    )

print("\nSaved:")
print(" -", OUT_CSV)
print(" -", OUT_XLSX)

print("\nOutput shape:", out_df.shape)

print("\nCaption length summary:")
condition_cols = [
    "N",
    "N+S",
    "N+SC",
    "N+SM",
    "N+SI",
    "N+SD",
    "N+SP",
    "N+SCMIDP",
]

for c in condition_cols:
    print(
        c,
        "mean length =",
        int(out_df[c].astype(str).str.len().mean()),
        "| unique =",
        out_df[c].nunique(),
    )

print("\nPreview:")
display(out_df.head())

# ============================================================
# 16. Download
# ============================================================

from google.colab import files

files.download(OUT_CSV)
files.download(OUT_XLSX)